TOPP Questionnare Chatbot

In [2]:
%pip install langgraph langchain langchain-openAI tavily-python graphviz matplotlib python-dotenv ipywidgets


Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os

# Specify the full path to your .env file
load_dotenv(r"C:\Users\Tendai\OneDrive\Chat Agents\.env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [5]:
from langchain_core.tools import tool
Questionnare = {"Question 1: Open sore, blister, or ulcer on either foot?": 
    "An open sore is a break in the skin that hasn't healed — \
    it may look like a raw patch, a wound, or a crater. A blister is \
    a bubble of fluid under the skin from friction or pressure. An ulcer \
    is a deeper, slower-healing wound often found on the bottom of the foot \
    in diabetic patients. You are looking for any spot on either foot where the skin is broken, \
    raw, or not intact.",
    "Question 2: Any new redness, warmth or swelling today?":
    "Redness means the skin looks pinker or redder than normal, \
    especially compared to the other foot. Warmth means an area feels noticeably hotter\
    to the touch than surrounding skin. Swelling means a part of the foot looks or feels \
    puffy and larger than usual. *New* means this wasn't there before — you are looking for \
    changes that appeared recently, not chronic conditions you've always had.",
    "Question 3: Any drainage, odor, or bleeding from the foot or toes?" :
    "Drainage means any fluid leaking from the skin — clear, yellow, or cloudy liquid coming \
    from a wound or between the toes. Odor means an unusual or unpleasant smell coming from the \
    foot, which can signal infection. Bleeding means any active or dried blood on the foot or \
    toes, even a small amount. You are looking for any sign that something is leaking or infected."}

#tool for foot check questionnaire
@tool
def foot_check(question_key: str) -> str:
    """
    Help users by returning in depth descriptions of the questions in the foot check questionnaire.
    Questions include: Question 1: Open sore, blister, or ulcer on either foot?
    
    Args: question_key (str): The users question, formatted as a key to the questionnnaire dictionary.
    """
    if question_key in Questionnare:
        return Questionnare[question_key]
    else:
        return ("Which question on the questionnaire can I help you with?")

    

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

SYSTEM_MSG = SystemMessage(content=
    "You are a questionnare assistant that helps users understand the questions \
    in a foot check questionnaire. You have access to a tool that provides in depth \
    descriptions of the questions."
)

llm = ChatOpenAI(model = 'gpt-4o-mini', temperature=0, openai_api_key=openai_api_key)
tools = [foot_check]
tool_node = ToolNode(tools)

#build agent
bound_llm = llm.bind_tools(tools)

#node functions
def llm_call(state: MessagesState):
    return {"messages": [bound_llm.invoke([SYSTEM_MSG] + state["messages"])]}
def should_continue(state: MessagesState):
    return "action" if state["messages"][-1].tool_calls else END
#nodes and edges
builder = StateGraph(MessagesState)
builder.add_node("llm_call", llm_call)
builder.add_node("environment", tool_node)
builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call", should_continue, {"action": "environment", END: END})
builder.add_edge("environment", "llm_call")

agent = builder.compile()


In [ ]:
user_input = input("Enter: ")
while user_input != "exit":
    agent.invoke({"messages": [HumanMessage(content=user_input)]})
    user_input = input("Enter: ")